In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [6]:
from pyspark.sql import SparkSession

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas 6 Big Data") \
    .getOrCreate()

# Membaca data
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_prd = spark.read.json("tugas6_produk.json")
df_rev = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

# Menampilkan jumlah baris & skema
print(f"Jumlah baris Transaksi : {df_trx.count()}")
df_trx.printSchema()

print(f"\nJumlah baris Produk    : {df_prd.count()}")
df_prd.printSchema()

print(f"\nJumlah baris Ulasan    : {df_rev.count()}")
df_rev.printSchema()

Jumlah baris Transaksi : 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)


Jumlah baris Produk    : 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)


Jumlah baris Ulasan    : 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [8]:
from pyspark.sql.functions import col

df_merged = df_trx \
    .join(df_prd, on="product_id", how="inner") \
    .join(df_rev, on="order_id", how="left")

df_transformed = df_merged.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga")
)

df_transformed.select("order_id", "nama_produk", "kategori", "unit_terjual", "harga", "total_pendapatan", "rating").show(5)

+--------+-----------+------------+------------+------+----------------+------+
|order_id|nama_produk|    kategori|unit_terjual| harga|total_pendapatan|rating|
+--------+-----------+------------+------------+------+----------------+------+
|     TX0|  Produk-21|  Elektronik|           2| 25000|           50000|     1|
|     TX1|   Produk-8|     Fashion|           6|250000|         1500000|  NULL|
|     TX2|  Produk-25|  Elektronik|           4| 25000|          100000|     2|
|     TX3|   Produk-3|     Fashion|           4| 75000|          300000|     5|
|     TX4|  Produk-19|Rumah Tangga|           6| 75000|          450000|  NULL|
+--------+-----------+------------+------------+------+----------------+------+
only showing top 5 rows



In [9]:
# 1. Buat kolom ada_ulasan sebelum na.fill()
df_enriched = df_transformed.withColumn(
    "ada_ulasan",
    col("rating").isNotNull()
)

# 2. Isi nilai null pada rating dengan angka 0
df_enriched = df_enriched.na.fill({"rating": 0})

# Tampilkan hasil
df_enriched.select("order_id", "nama_produk", "rating", "ada_ulasan").show(10)

+--------+-----------+------+----------+
|order_id|nama_produk|rating|ada_ulasan|
+--------+-----------+------+----------+
|     TX0|  Produk-21|     1|      true|
|     TX1|   Produk-8|     0|     false|
|     TX2|  Produk-25|     2|      true|
|     TX3|   Produk-3|     5|      true|
|     TX4|  Produk-19|     0|     false|
|     TX5|  Produk-10|     5|      true|
|     TX6|  Produk-26|     4|      true|
|     TX7|   Produk-4|     0|     false|
|     TX8|   Produk-7|     5|      true|
|     TX9|  Produk-30|     0|     false|
+--------+-----------+------+----------+
only showing top 10 rows



In [11]:
# Buat folder HDFS tujuan (sesuaikan nama user jika diperlukan)
!hdfs dfs -mkdir -p /user/mahasiswa/tugas6

path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"

# Simpan ke HDFS terpartisi berdasarkan kategori
df_enriched.write.mode("overwrite").partitionBy("kategori").parquet(path_hdfs)
print("Penyimpanan ke HDFS selesai.")

# Verifikasi struktur folder di HDFS
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

# Baca kembali untuk verifikasi jumlah data
df_hdfs_verify = spark.read.parquet(path_hdfs)
print(f"Verifikasi sukses! Total baris di HDFS: {df_hdfs_verify.count()}")

2026-09-28 11:56:07,048 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Penyimpanan ke HDFS selesai.
2026-09-28 11:56:09,063 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
-rw-r--r--   3 aret supergroup          0 2026-09-28 11:56 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - aret supergroup          0 2026-09-28 11:56 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 aret supergroup      16364 2026-09-28 11:56 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-a3e92d47-b9a3-4484-a764-3ea40c9c777c.c000.snappy.parquet
drwxr-xr-x   - aret supergroup          0 2026-09-28 11:56 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 aret supergroup      11134 2026-09-28 11:56 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-a3e92d47-b9a3-4484-a764-3ea40c9c777c.c000

In [13]:
from pyspark.sql.functions import count, when, col, round as spark_round

# Menghitung persentase transaksi yang memiliki ulasan per kategori
df_insight = df_hdfs_verify.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"),
    count(when(col("ada_ulasan") == True, True)).alias("transaksi_ada_ulasan")
).withColumn(
    "persentase_ulasan",
    spark_round((col("transaksi_ada_ulasan") / col("total_transaksi")) * 100, 2)
).orderBy("persentase_ulasan")

df_insight.show()

+------------+---------------+--------------------+-----------------+
|    kategori|total_transaksi|transaksi_ada_ulasan|persentase_ulasan|
+------------+---------------+--------------------+-----------------+
|     Makanan|            536|                 369|            68.84|
|   Kesehatan|            961|                 662|            68.89|
|     Fashion|           1035|                 726|            70.14|
|Rumah Tangga|            815|                 575|            70.55|
|  Elektronik|           1653|                1168|            70.66|
+------------+---------------+--------------------+-----------------+

